In [1]:
import torch
import sys
print("Interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Interpreter: d:\maga25\VKRTimeSeries\.venv\Scripts\python.exe
CUDA available: True
GPU name: NVIDIA GeForce GTX 1660


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if device.type == 'cuda':
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"Всего видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce GTX 1660
Всего видеопамяти: 6.44 GB


In [3]:
# Standard
import random

import numpy as np
import pandas as pd
import torch

# Third Party
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
)

# First Party
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

d:\maga25\VKRTimeSeries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

------------------------------------------------------------------------


Дообучение на датасете metr-la

In [8]:
import h5py
import pandas as pd
import numpy as np

file_path = 'data/metr-la.h5'

with h5py.File(file_path, 'r') as f:
    # Данные лежат в группе 'df'
    group = f['df']
    
    data = group['block0_values'][:]          # (времена, сенсоры)
    columns_raw = group['block0_items'][:]    # названия сенсоров
    
    # Преобразуем названия столбцов в строки
    if columns_raw.dtype.type is np.bytes_:
        columns = [col.decode('utf-8') for col in columns_raw]
    else:
        columns = list(columns_raw)
    
    # Пытаемся взять индекс из axis0
    index_raw = group['axis0'][:]
    if len(index_raw) == data.shape[0]:
        if index_raw.dtype.type is np.bytes_:
            index = [idx.decode('utf-8') for idx in index_raw]
        else:
            index = index_raw
        print(f"Используем axis0 как индекс (длина {len(index)})")
    else:
        print(f"axis0 имеет длину {len(index_raw)}, но данных {data.shape[0]} — не используем")
        index = None
    
    # Создаём DataFrame
    if index is None:
        df = pd.DataFrame(data, columns=columns)
    else:
        df = pd.DataFrame(data, columns=columns, index=index)

print("Размер данных:", df.shape)
print("Первые 5 строк:")
print(df.head())

axis0 имеет длину 207, но данных 34272 — не используем
Размер данных: (34272, 207)
Первые 5 строк:
      773869     767541     767542     717447     717446     717445  773062  \
0  64.375000  67.625000  67.125000  61.500000  66.875000  68.750000  65.125   
1  62.666667  68.555556  65.444444  62.444444  64.444444  68.111111  65.000   
2  64.000000  63.750000  60.000000  59.000000  66.500000  66.250000  64.500   
3   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000   0.000   
4   0.000000   0.000000   0.000000   0.000000   0.000000   0.000000   0.000   

   767620     737529     717816  ...     772167  769372     774204     769806  \
0  67.125  59.625000  62.750000  ...  45.625000  65.500  64.500000  66.428571   
1  65.000  57.444444  63.333333  ...  50.666667  69.875  66.666667  58.555556   
2  64.250  63.875000  65.375000  ...  44.125000  69.000  56.500000  59.250000   
3   0.000   0.000000   0.000000  ...   0.000000   0.000   0.000000   0.000000   
4   0.000   0.000000 